In [44]:
import os
import json
import time
import re
import difflib
from textwrap import dedent
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI

DPO_DATASET_DIR = "./finetuning_data/crm-dpo-dataset/"
SFT_DATASET_DIR = "./finetuning_data/crm-sft-dataset/"

PERSONAS_PATH = "./data/personas.json"
os.makedirs(SFT_DATASET_DIR, exist_ok=True)

persona_tokens = []
if os.path.exists(PERSONAS_PATH):
    with open(PERSONAS_PATH, "r", encoding="utf-8") as f:
        personas = json.load(f)
    persona_tokens = sorted({p.get("name", "") for p in personas if p.get("name")})


# 기존 DPO 데이터셋 파일 읽기
with open(os.path.join(DPO_DATASET_DIR, "cycle_01_v2.json"), "r") as f:
    data = json.load(f)
    print(f"로드된 DPO 데이터 수 : {len(data)}")
    print(f"DPO 데이터 컬럼 들 : {list(data[0].keys())}")
    print(data[:10])
    print(type(data), type(f))

# SFT용 추가 프롬프트 정의
output_template = dedent(f"""
아래 출력 규칙과 컨텍스트를 바탕으로 CRM 메시지를 재작성하라.
[출력 규칙]
- 다음 요소는 포함하지 않는다:
  1) 영어/한국어의 어색한 혼용 (브랜드/제품 고유명 제외, 예: 'everyday 사용')
  2) 페르소나/개인정보의 직접 호명 (예: 'Budget_Seeker님')
  3) 과도한 특수문자, 이모지, 구분선

[출력 형식]
제목:title 내용
본문:body 내용
""")

MODEL_NAME = "gpt-5-nano"
BATCH_SIZE = 1
MAX_OUTPUT_TOKENS = 1024
RETRY_LIMIT = 3

DEBUG = False
DEBUG_MAX_CHARS = 500

MIN_TITLE_LEN = 4
MIN_BODY_LEN = 15

client = OpenAI()

def _debug(message):
    if DEBUG:
        print(message)

def _response_text(response):
    if hasattr(response, "output_text"):
        text = response.output_text
        if isinstance(text, list):
            return "\n".join(text)
        return text
    try:
        parts = []
        for item in response.output:
            if getattr(item, "type", None) == "message":
                for content in item.content:
                    if hasattr(content, "text"):
                        parts.append(content.text)
        return "\n".join(parts)
    except Exception:
        return ""

def rule_based_clean(text, persona_tokens):
    if not text:
        return text
    cleaned = text
    cleaned = re.sub(r"```[a-zA-Z]*", "", cleaned)
    cleaned = cleaned.replace("```", "")
    cleaned = re.sub(r"(?m)^\s*(json|text)\s*$", "", cleaned)
    cleaned = cleaned.translate(str.maketrans("", "", "{}[]"))
    cleaned = re.sub(r"\"?title\"?\s*:", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\"?body\"?\s*:", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\[\s*제목\s*\]|\[\s*본문\s*\]|CTA\s*:", "", cleaned)
    if persona_tokens:
        pattern = r"(?<!\w)(?:" + "|".join(map(re.escape, persona_tokens)) + r")(?:님)?(?!\w)"
        cleaned = re.sub(pattern, "", cleaned)
    cleaned = re.sub(r"\S+님을\s*위해", "", cleaned)
    cleaned = re.sub(r"\S+분들께", "", cleaned)
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()

def extract_removed_parts(original, cleaned, limit=5):
    matcher = difflib.SequenceMatcher(None, original, cleaned)
    parts = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag in ("delete", "replace"):
            removed = original[i1:i2]
            removed = re.sub(r"\s+", " ", removed).strip()
            if removed:
                if len(removed) > 80:
                    removed = removed[:77] + "..."
                parts.append(removed)
    seen = set()
    uniq = []
    for part in parts:
        if part not in seen:
            seen.add(part)
            uniq.append(part)
    return uniq[:limit]

def _min_len_ok(text, min_len):
    if text is None:
        return False
    compact = re.sub(r"\s+", "", text)
    return len(compact) >= min_len

def normalize_refined(text):
    if not text:
        return ""
    cleaned = text.strip()
    cleaned = re.sub(r"```[a-zA-Z]*", "", cleaned)
    cleaned = cleaned.replace("```", "").strip()
    title = None
    body = None
    # Try JSON extraction first
    if "{" in cleaned and "}" in cleaned:
        start = cleaned.find("{")
        end = cleaned.rfind("}") + 1
        try:
            obj = json.loads(cleaned[start:end])
            if isinstance(obj, dict):
                title = str(obj.get("title", "")).strip() or None
                body = str(obj.get("body", "")).strip() or None
        except Exception:
            pass
    # Fallback to label lines
    if title is None or body is None:
        lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
        for line in lines:
            if line.startswith("제목:"):
                title = line.split("제목:", 1)[1].strip() or title
            elif line.lower().startswith("title:"):
                title = line.split(":", 1)[1].strip() or title
            elif line.startswith("본문:"):
                body = line.split("본문:", 1)[1].strip() or body
            elif line.lower().startswith("body:"):
                body = line.split(":", 1)[1].strip() or body
        if title is None and lines:
            title = lines[0]
        if body is None and len(lines) > 1:
            body = lines[1]
    if not _min_len_ok(title, MIN_TITLE_LEN) or not _min_len_ok(body, MIN_BODY_LEN):
        return ""
    result = []
    result.append(f"제목: {title}")
    result.append(f"본문: {body}")
    return "\n".join(result).strip()

def refine_message(context, draft, output_rules):
    system = output_rules.strip()
    user = f"{context}\n\n{draft}\n"
    last_error = None
    for attempt in range(RETRY_LIMIT):
        try:
            _debug(f"[GPT] attempt={attempt + 1}/{RETRY_LIMIT} model={MODEL_NAME}")
            _debug(f"[GPT] system_len={len(system)} user_len={len(user)}")
            _debug(f"[GPT] system_preview=\n{system[:DEBUG_MAX_CHARS]}")
            _debug(f"[GPT] user_preview=\n{user[:DEBUG_MAX_CHARS]}")
            response = client.responses.create(
                reasoning={"effort": "low"},
                model=MODEL_NAME,
                input=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )
            _debug(f"[GPT] response_status={getattr(response, 'status', None)} incomplete={getattr(response, 'incomplete_details', None)}")
            text = _response_text(response).strip()
            text = normalize_refined(text)
            _debug(f"[GPT] output_text_len={len(text)}")
            _debug(f"[GPT] output_text_preview=\n{text[:DEBUG_MAX_CHARS]}")
            if DEBUG:
                try:
                    dump = response.model_dump()
                    _debug(f"[GPT] response_dump_preview=\n{json.dumps(dump, ensure_ascii=False)[:DEBUG_MAX_CHARS]}")
                except Exception:
                    _debug(f"[GPT] response_repr_preview=\n{str(response)[:DEBUG_MAX_CHARS]}")
            if text:
                return text
        except Exception as exc:
            last_error = exc
            _debug(f"[GPT] error={type(exc).__name__}: {exc}")
            time.sleep(2 ** attempt)
    if last_error:
        raise last_error
    return ""


로드된 DPO 데이터 수 : 1248
DPO 데이터 컬럼 들 : ['prompt', 'chosen', 'rejected', 'best_index', 'rejected_index', 'reason_best', 'reason_rejected']
[{'prompt': '[컨텍스트]\n- Persona: Budget_Seeker (가격 대비 효율, 실사용 후기, 복합성/지성, 가벼운 트러블, 가성비 중심, 쿠폰·세일 적극 활용, 브랜드 충성도 낮음)\n- Stage: Referral\n- Brand/Product: 에뛰드 / 뽀오얀 미소 발효 립&아이 리무버 250ml (대용량)\n- Price: 12,750원\n- Event: 없음', 'chosen': '```json\n{\n  "title": "[뽀오얀 미소 발효 립&아이 리무버] efficacy & affordable 선택 가이드",\n  "body": "EDDY Range의 포어 발효 기술로 눈 clarity 유지가 핵심입니다! 대용량 250ml로 everyday 사용부터 여행까지 편리하게 쓸 수 있어요. 지성/복합성 피부도 부담 없이 사용 가능하고, 기존 제품 대비 비용 대비 성능이 뛰어나다는 후기 다수가 있습니다. 오늘부터 10일간만 특가로 판매 중이니 서둘러 보세요! 함께 써줄 친구에게 추천해주면 추가 혜택도 받을 수 있답니다."\n}\n```', 'rejected': '```json\n{\n  "title": "⌛ 혜택 종료 전 마지막 추천 기회",\n  "body": "[브랜드명]에서 추천해준 Olive Kit이 정말 좋아요! 가볍지만 효과적인 입술&눈 리무버로 하루 종일 사용하실 수 있어요. 여러 번 쓸 수 있어 부담 없이 사용할 수 있답니다. 오늘도 같이 써볼까요? 👇"\n}\n```', 'best_index': 2, 'rejected_index': 0, 'reason_best': '가장 간결하고 필요한 정보만 담아 브랜드명 노출 최소화, 페르소나 호명 없이도 메시지의 가치와 혜택이 명확히 전달된

In [48]:
# 데이터 갯수 확인 이후 남은 데이터 refine
with open(os.path.join(SFT_DATASET_DIR, "cycle_01_v2.jsonl"), "r") as f:
    remain_data_num = sum(1 for _ in f)
    print(f"저장된 SFT 데이터 수 : {remain_data_num}")

output_template_for_sft = dedent(f"""
[출력 규칙]
- 다음 요소는 포함하지 않는다:
  1) 영어/한국어의 어색한 혼용 (브랜드/제품 고유명 제외, 예: 'everyday 사용')
  2) 페르소나/개인정보의 직접 호명 (예: 'Budget_Seeker님')
  3) 과도한 특수문자, 이모지, 구분선

[출력 형식]
제목:
본문:
""")

total_batches = (len(data[remain_data_num:]) + BATCH_SIZE - 1) // BATCH_SIZE
for batch_index in range(total_batches):
    refined_data = []
    start = batch_index * BATCH_SIZE + remain_data_num
    batch = data[start : start + BATCH_SIZE]
    print(f"Batch {batch_index + 1}/{total_batches}")

    for idx_in_batch, example in enumerate(batch):
        sample_index = start + idx_in_batch + 1
        context = example.get("prompt", "")
        draft = example.get("chosen", "")
        cleaned = rule_based_clean(draft, persona_tokens)
        if cleaned != draft:
            removed_parts = extract_removed_parts(draft, cleaned)
            if removed_parts:
                print(f"[정제됨] {sample_index}번: " + ", ".join(removed_parts))
            else:
                print(f"[정제됨] {sample_index}번: 변경 감지(공백/형식)")
        refined = refine_message(context, cleaned, output_template)
        if not refined:
            print(f"[제외] {sample_index}번: 제목/본문 길이 부족")
            continue
        refined_data.append({**example, "chosen_refined": refined})

    with open(os.path.join(SFT_DATASET_DIR, "cycle_01_v2.jsonl"), "a") as f:
        for example in refined_data:
            prompt = example.get("prompt", "")
            chosen = example.get("chosen_refined", example.get("chosen", ""))
            prompt = "다음 조건에 맞는 CRM 메시지를 작성하세요. " + prompt + f" {output_template_for_sft.replace('\n', ' ')}"
            f.write(json.dumps({"prompt": prompt, "chosen": chosen}, ensure_ascii=False) + "\n")

저장된 SFT 데이터 수 : 116
Batch 1/1132
[정제됨] 117번: ```json { "title": "[, ], "body":, } ```
Batch 2/1132
[정제됨] 118번: ```json { "title": "[, ], "body":, } ```
Batch 3/1132
[정제됨] 119번: ```json { "title": "[, ], "body":, } ```
Batch 4/1132
[정제됨] 120번: ```json { "title": "[, ], "body":, } ```
Batch 5/1132
[정제됨] 121번: ```json { "title":, "body":, } ```
Batch 6/1132
[정제됨] 122번: ```json { "title":, "body":, } ```
Batch 7/1132
[정제됨] 123번: ```json { "title":, "body":, } ```
Batch 8/1132
[정제됨] 124번: ```json { "title": "[, ], "body":, } ```
Batch 9/1132
[정제됨] 125번: ```json { "title": "[, ], "body":, } ```
Batch 10/1132
[정제됨] 126번: ```json { "title": "[, ], "body":, } ```
Batch 11/1132
[정제됨] 127번: ```json { "title":, body": "[, ], } ```
Batch 12/1132
[정제됨] 128번: ```json { "title":, body": "[, ], } ```
Batch 13/1132
[정제됨] 129번: ```json { "title":, body": "[, ], } ```
Batch 14/1132
[정제됨] 130번: ```json { "title": "[, ], "body":, } ```
Batch 15/1132
[정제됨] 131번: ```json { "title": "[, ], "body":, } ```
Batch